# Proxy De-risk Track — Qwen-local ICRL + Axis (Colab)

Runs the **Week 1 proxy path** on a single GPU runtime:

1. Generate ICRL conversations with **local Qwen3-8B** (no Anthropic API)
2. Extract activations → build **proxy axis** (`value_axis_proxy.npy`)
3. Loose gate: L21/L22 AUROC ≥ **0.75** (NOT faithful 0.93)

**Label:** proxy axis for noise feasibility only — not faithful Stage 1.

**Runtime:** A100 strongly recommended. ICRL generation is the bottleneck.

## Run it in two passes
- **Pass 1 (pilot, N=10):** sanity check only — confirms generation works and the
  conversations look sane. The gate at N=10 is statistically meaningless and will
  almost certainly say FAIL. That is expected; do not react to the pilot number.
- **Pass 2 (full, N=100):** set `N = 100` and re-run from the generate cell. This is
  the real proxy gate.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable GPU: Runtime → Change runtime type → GPU'
print(torch.cuda.get_device_name(0))

In [ ]:
# Pull the latest code (must include today's JSON-robustness fixes).
!git clone https://github.com/abdelmagid07/latent_failiure_prediction.git
%cd latent_failiure_prediction/stage1
!git log --oneline -1
!pip install -q -e .

In [ ]:
# Qwen3-8B is not gated, so HF login is usually unnecessary. Uncomment if needed.
# from huggingface_hub import login; login()

In [ ]:
# Pilot first (N=10). For the real gate, set N = 100 and re-run from here.
N = 10
OUTPUT = 'data/icrl_proxy_pilot.json' if N <= 10 else 'data/icrl_proxy.json'

!python -m stage1.icrl_gen.generate --n {N} --backend local_qwen \
  --output {OUTPUT} --resume --max-turn-retries 8

### Inspect before spending GPU time on extraction

Eyeball one generated conversation. Check that: pre-discovery turns pursue a *wrong*
hypothesis (and get `-1`), the discovery turn satisfies the criterion (`+1`), and the
first post-discovery turn is a clean rewrite with sensible `satisfying_char_start`.
If conversations look degenerate (empty paragraphs, no thinking, judge nonsense),
fix that before scaling to N=100.

In [ ]:
from pathlib import Path
from stage1.icrl.schema import load_conversations

convs = load_conversations(Path(OUTPUT))
print(f'Generated {len(convs)} / {N} requested (skips are normal when a turn fails repeatedly).')
assert convs, 'No conversations generated — check the generation log above.'

c = convs[0]
print(f'\nconv_id={c.conv_id}  criterion={c.criterion_id}  discovery_idx={c.discovery_paragraph_idx}')
print(f'first_post_discovery_turn_idx={c.first_post_discovery_turn_idx}  satisfying_char_start={c.satisfying_char_start}')
for i, t in enumerate(c.turns):
    mark = '   <-- FIRST POST-DISCOVERY (axis is built from this paragraph)' if i == c.first_post_discovery_turn_idx else ''
    body = t.content if len(t.content) <= 500 else t.content[:500] + ' ...'
    print(f'\n[{i}] {t.role}{mark}:\n{body}')

In [ ]:
!python -m stage1.pipeline.extract_activations --icrl {OUTPUT} \
  --activations-dir data/activations_proxy --force

In [ ]:
# Loose 0.75 gate. At N=10 this is a wiring check, NOT a real gate — FAIL is expected.
# Only the N=100 result is meaningful.
!python -m stage1.pipeline.run_proxy_gate --icrl {OUTPUT} --skip-extract

In [ ]:
# Download the proxy artifacts (run after the N=100 pass).
from google.colab import files
for p in ['data/value_axis_proxy.npy', 'data/axis_manifest_proxy.json', 'data/auroc_by_layer_proxy.png']:
    files.download(p)